Overwrite this cell with Notebook Metadata

In [1]:
import json
import textwrap
from pathlib import Path

ROOT = Path('/home/xiaoruiwang/data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis')
TUTORIAL_DIR = ROOT / 'teaching_demos/4.reaction_condition_tutorial/ARGCN'
TUTORIAL_DIR.mkdir(parents=True, exist_ok=True)


def _source(text: str):
    text = textwrap.dedent(text).strip('\n')
    if not text:
        return []
    return [line + '\n' for line in text.splitlines()]


def md(text: str):
    return {
        'cell_type': 'markdown',
        'metadata': {},
        'source': _source(text),
    }


def code(text: str):
    return {
        'cell_type': 'code',
        'execution_count': None,
        'metadata': {},
        'outputs': [],
        'source': _source(text),
    }


def notebook(cells):
    return {
        'cells': cells,
        'metadata': {
            'kernelspec': {
                'display_name': 'Python 3',
                'language': 'python',
                'name': 'python3',
            },
            'language_info': {
                'name': 'python',
                'version': '3.11',
            },
        },
        'nbformat': 4,
        'nbformat_minor': 5,
    }


env_cells = [
    md('''
    # ARGCN 环境配置教程

    ## 概述

    这份 notebook 是 ARGCN 教学系列的第 1 部分。这里的 **ARGCN** 指代本教程对 `reaction-gcnn` 中
    `train_attention_model.py + models/relgcn.py + models/attention_readout.py` 这一条主线做出的
    **PyTorch 教学复现版本**。

    原仓库使用的是 **Chainer + chainer-chemistry**。为了适配当前更常见、也更易维护的环境，本教程不再要求读者安装
    Chainer，而是使用 **conda + PyTorch + RDKit** 复现同样的数据流和模型结构。

    ### 本 notebook 内容

    | 章节 | 内容 |
    |------|------|
    | 1 | 教学版依赖设计：为什么从 Chainer 迁移到 Torch |
    | 2 | 使用 conda 创建独立环境 |
    | 3 | 注册 Jupyter kernel |
    | 4 | 验证 Torch / RDKit / 数据文件路径 |
    | 5 | 对照原仓库查看源码结构 |
    '''),
    md('''
    ## 1. 核心依赖一览

    原仓库的时间背景是 2020 年前后，因此依赖栈明显偏旧。教学版不追求复现旧环境，而是复现**算法主线**：

    | 模块 | 原仓库实现 | 教学版实现 |
    |------|------------|------------|
    | 深度学习框架 | Chainer | PyTorch |
    | 分子处理 | RDKit | RDKit |
    | 图网络实现 | chainer-chemistry | 纯 Torch 最小复现 |
    | notebook 环境 | 未约束 | conda 独立环境 |

    > 说明
    >
    > 为了适配“当前 channel 下的最新兼容版本”，下面的 conda 安装命令只固定 `python=3.11`，其余核心包交给 channel 解出当前最新兼容组合。
    > 这样比硬编码一组很快过时的精确小版本更稳妥。
    '''),
    code('''
    import os
    from pathlib import Path

    # ====== Step 1: 定位项目根目录与关键路径 ======
    def find_project_root(start=None):
        here = Path(start or os.getcwd()).resolve()
        for path in [here, *here.parents]:
            if (path / 'teaching_demos').exists() and (path / 'source_repos').exists():
                return path
        raise RuntimeError('未找到项目根目录，请确认当前 notebook 位于 Chemical_Synthesis 项目内。')

    PROJECT_ROOT = find_project_root()
    TUTORIAL_DIR = PROJECT_ROOT / 'teaching_demos' / '4.reaction_condition_tutorial' / 'ARGCN'
    REPO_DIR = PROJECT_ROOT / 'source_repos' / 'reaction-gcnn'
    ENV_DIR = PROJECT_ROOT / 'envs' / 'argcn_tutorial_envs'
    DEMO_DATA = TUTORIAL_DIR / 'data' / 'demo_data.csv'

    print(f'PROJECT_ROOT = {PROJECT_ROOT}')
    print(f'TUTORIAL_DIR = {TUTORIAL_DIR}')
    print(f'REPO_DIR = {REPO_DIR}')
    print(f'ENV_DIR = {ENV_DIR}')
    print(f'DEMO_DATA = {DEMO_DATA}')
    '''),
    md('''
    ## 2. 使用 conda 创建教学环境

    ### 为什么这里必须使用 conda？

    1. RDKit 在 conda-forge 上的安装最稳定。
    2. PyTorch 可以通过 `pytorch` channel 直接拿到当前兼容版本。
    3. 读者后续还会在 notebook 中切换 kernel，conda 比 `venv` 更适合这类科研工作流。

    下面这段命令会在项目根目录下创建：`envs/argcn_tutorial_envs`
    '''),
    code('''
    %%bash -s "$ENV_DIR"

    ENV_DIR="$1"

    if ! command -v conda >/dev/null 2>&1; then
      FOUND_CONDA_SH=""
      for candidate in "$HOME/miniconda3/etc/profile.d/conda.sh" "$HOME/anaconda3/etc/profile.d/conda.sh" "/opt/conda/etc/profile.d/conda.sh"; do
        if [ -f "$candidate" ]; then
          FOUND_CONDA_SH="$candidate"
          break
        fi
      done

      if [ -z "$FOUND_CONDA_SH" ]; then
        echo "未找到 conda，请先安装 Miniconda 或 Anaconda。"
        exit 1
      fi
      source "$FOUND_CONDA_SH"
    else
      source "$(conda info --base)/etc/profile.d/conda.sh"
    fi

    conda create -y -p "$ENV_DIR" python=3.11
    conda install -y -p "$ENV_DIR" -c pytorch -c conda-forge \
      pytorch cpuonly rdkit pandas numpy matplotlib scikit-learn jupyterlab ipykernel tqdm

    echo "==> conda 环境创建完成: $ENV_DIR"
    '''),
    md('''
    ### 可选：如果你后续需要 GPU

    教学 notebook 默认采用 CPU 方案，因为它最容易复现。如果你确实要在 GPU 上跑，可以把上面的 `cpuonly`
    替换成 PyTorch 官方安装矩阵中当前推荐的 `pytorch-cuda` 组合。

    教学逻辑本身与 CPU / GPU 无关。
    '''),
    md('''
    ## 3. 注册为 Jupyter Kernel（推荐）

    这样做的目的是让后续 2 份 notebook 都可以直接切换到同一个教学环境。
    '''),
    code('''
    %%bash -s "$ENV_DIR"

    ENV_DIR="$1"
    source "$ENV_DIR/bin/activate"
    python -m ipykernel install --user --name argcn_tutorial --display-name "ARGCN Tutorial"
    echo "==> Kernel 'ARGCN Tutorial' 已注册完成"
    '''),
    md('''
    ## 4. 验证环境

    > 注意
    >
    > 在执行下面的验证代码之前，请先把 notebook kernel 切换到 `ARGCN Tutorial`。
    '''),
    code('''
    # ====== Step 4.1: 验证核心依赖 ======
    import sys
    import numpy as np
    import pandas as pd
    import torch
    import sklearn
    from rdkit import rdBase

    print(f'Python   : {sys.version.split()[0]}')
    print(f'NumPy    : {np.__version__}')
    print(f'pandas   : {pd.__version__}')
    print(f'PyTorch  : {torch.__version__} (CUDA available: {torch.cuda.is_available()})')
    print(f'sklearn  : {sklearn.__version__}')
    print(f'RDKit    : {rdBase.rdkitVersion}')
    '''),
    md('''
    ## 5. 验证 RDKit 与 Torch 的基础图操作

    这个检查不是为了“训练模型”，而是为了确认后续两份 notebook 依赖的两个核心步骤都可用：

    1. RDKit 能把 SMILES 变成分子对象。
    2. Torch 能对邻接矩阵和节点特征做张量运算。
    '''),
    code('''
    # ====== Step 5.1: RDKit 分子解析 + Torch 张量运算 ======
    from rdkit import Chem

    smiles = 'c1ccccc1Br'
    mol = Chem.MolFromSmiles(smiles)
    atom_ids = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
    n = len(atom_ids)

    adj = torch.zeros((n, n), dtype=torch.float32)
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        adj[i, j] = 1.0
        adj[j, i] = 1.0

    x = torch.tensor(atom_ids, dtype=torch.float32).unsqueeze(-1)
    aggregated = adj @ x

    print('SMILES:', smiles)
    print('atom_ids:', atom_ids)
    print('adj shape:', tuple(adj.shape))
    print('x shape:', tuple(x.shape))
    print('adj @ x shape:', tuple(aggregated.shape))
    print('aggregated[:5]:')
    print(aggregated[:5])
    '''),
    md('''
    ## 6. `reaction-gcnn` 源码结构概览

    这一步不是装饰，而是后面两份教学 notebook 的地图。

    ### 本教程重点对应的源码文件

    | 文件 | 作用 |
    |------|------|
    | `source_repos/reaction-gcnn/train.py` | 基础版三分子图拼接模型 |
    | `source_repos/reaction-gcnn/train_attention_model.py` | 带 reaction-level attention 的版本 |
    | `source_repos/reaction-gcnn/models/relgcn.py` | 关系图卷积编码器 |
    | `source_repos/reaction-gcnn/models/attention_readout.py` | 三分子级别的注意力聚合 |
    | `source_repos/reaction-gcnn/dataset/suzuki_data_frame_parser.py` | 数据解析与标签向量构造 |
    '''),
    code('''
    # ====== Step 6.1: 打印关键源码树 ======
    from pathlib import Path

    key_paths = [
        REPO_DIR / 'train.py',
        REPO_DIR / 'train_attention_model.py',
        REPO_DIR / 'models' / 'relgcn.py',
        REPO_DIR / 'models' / 'attention_readout.py',
        REPO_DIR / 'models' / 'classifier.py',
        REPO_DIR / 'dataset' / 'suzuki_data_frame_parser.py',
        DEMO_DATA,
    ]

    for path in key_paths:
        print(path.relative_to(PROJECT_ROOT), '->', path.exists())
    '''),
    md('''
    ## 7. 快速理解 ARGCN 教学版的工作流

    ```text
    demo_data.csv
        ↓
    反应 SMILES 拆分为 Reactant1 / Reactant2 / Product
        ↓
    条件名称映射为全局多热标签向量
        ↓
    每个分子转成 (atom_ids, relation_adjs)
        ↓
    共享 RelGCN 编码器
        ↓
    reaction-level attention readout
        ↓
    多标签条件预测 logits
    ```

    这也是后面两份 notebook 的顺序：

    - `2_数据处理.ipynb`: 负责把原始文本变成图张量和标签向量
    - `3_模型展示.ipynb`: 负责把这些张量送入 Torch 版 ARGCN，演示推理路径
    '''),
    md('''
    ## 8. 总结

    这一份 notebook 完成了三件事：

    1. 用 **conda** 明确了教学环境的构建方式。
    2. 把原始 Chainer 工程替换成了更适合当前学习和维护的 **Torch 教学环境**。
    3. 给后续两份 notebook 标出了准确的源码映射路径。

    下一步建议直接打开 `2_数据处理.ipynb`，先把 `reaction-gcnn` 的三分子输入和多标签条件向量看懂。
    '''),
]

data_cells = [
    md('''
    # ARGCN 数据处理教程

    ## 概述

    本 notebook 展示 ARGCN 教学版的**完整数据处理流程**。这里的目标不是“黑盒调用原仓库 parser”，而是把
    `reaction-gcnn/dataset/suzuki_data_frame_parser.py` 背后的关键步骤拆开，用最小化的 Python + RDKit 代码重新说明：

    1. 原始反应记录长什么样
    2. 反应如何拆成 `Reactant1 / Reactant2 / Product`
    3. 条件名称如何编码成多热标签向量
    4. 分子如何转成关系图卷积需要的 `atom_ids + relation_adjs`
    5. 多个样本如何拼成模型输入 batch

    ### 本 notebook 内容

    | 章节 | 内容 |
    |------|------|
    | 1 | 读取微型教学数据集 |
    | 2 | 对照 Suzuki 字典构造标签空间 |
    | 3 | 解析反应 SMILES |
    | 4 | 生成仓库风格的标签字段 |
    | 5 | 把每个分子转成关系图张量 |
    | 6 | 构造 ARGCN batch |
    '''),
    code('''
    # ====== Step 0: 环境初始化 ======
    import os
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import torch
    from rdkit import Chem

    def find_project_root(start=None):
        here = Path(start or os.getcwd()).resolve()
        for path in [here, *here.parents]:
            if (path / 'teaching_demos').exists() and (path / 'source_repos').exists():
                return path
        raise RuntimeError('未找到项目根目录')

    PROJECT_ROOT = find_project_root()
    TUTORIAL_DIR = PROJECT_ROOT / 'teaching_demos' / '4.reaction_condition_tutorial' / 'ARGCN'
    REPO_DIR = PROJECT_ROOT / 'source_repos' / 'reaction-gcnn'
    DEMO_DATA = TUTORIAL_DIR / 'data' / 'demo_data.csv'
    DICT_PATH = REPO_DIR / 'data' / 'all_dictionaries' / 'suzuki_dict.csv'
    REPO_STYLE_CSV = TUTORIAL_DIR / 'data' / 'demo_repo_style.csv'

    print('DEMO_DATA    =', DEMO_DATA)
    print('DICT_PATH    =', DICT_PATH)
    print('REPO_STYLE_CSV =', REPO_STYLE_CSV)
    '''),
    md('''
    ## 1. 读取微型教学数据集

    ### 为什么先看这个表？

    原仓库最终读入的是已经清洗好的 CSV，但真正的教学起点应该更早：

    - 一条反应记录包含 `reaction_smiles`
    - 条件列还是**人类可读的名称**
    - 多溶剂、多添加剂会以 `|` 分隔

    这更接近你自己手头实验数据的初始形态。

    ### 源码对应

    - 数据入口: `source_repos/reaction-gcnn/train.py`
    - 解析器: `source_repos/reaction-gcnn/dataset/suzuki_csv_file_parser.py`
    - 真正的逻辑主体: `source_repos/reaction-gcnn/dataset/suzuki_data_frame_parser.py`
    '''),
    code('''
    # ====== Step 1: 读取微型教学数据 ======
    demo_df = pd.read_csv(DEMO_DATA)
    demo_df
    '''),
    md('''
    ## 2. 读取 Suzuki 条件字典并构造全局标签空间

    `reaction-gcnn` 不是分别做“金属分类”“配体分类”“碱分类”……，而是把这些条件家族拼接成一个**统一的多热向量**。

    对 Suzuki 数据，`train.py` 中给出的家族规模是：

    - 金属 `M`: 28
    - 配体 `L`: 23
    - 碱 `B`: 35
    - 溶剂 `S`: 10
    - 添加剂 `A`: 17

    总计 113 个真实条件槽位。原 parser 还会额外保留：

    - 1 个保留槽位（教学上可以理解为“预留/unknown”）
    - 每个条件家族 1 个“空条件”槽位，共 5 个

    因此最终 `class_num = 119`。

    ### 源码对应

    - `source_repos/reaction-gcnn/train.py` 中的 `class_dict`
    - `source_repos/reaction-gcnn/dataset/suzuki_data_frame_parser.py` 中的 `rea_cat` 构造逻辑
    '''),
    code('''
    # ====== Step 2: 构造全局标签映射 ======
    suzuki_dict_df = pd.read_csv(DICT_PATH)

    FAMILY_META = {
        'M': ('metal_name', 'metal_bin'),
        'L': ('ligand_name', 'ligand_bin'),
        'B': ('base_name', 'base_bin'),
        'S': ('solvent_name', 'solvent_bin'),
        'A': ('additive_name', 'additive_bin'),
    }
    RAW_COL_TO_FAMILY = {
        'metal': 'M',
        'ligand': 'L',
        'base': 'B',
        'solvent': 'S',
        'additive': 'A',
    }

    def extract_family_entries(df, family, name_col, bin_col):
        part = df[[name_col, bin_col]].dropna().copy()
        part[bin_col] = part[bin_col].astype(str)
        part = part[part[bin_col].str.startswith(family)]
        part['local_index'] = part[bin_col].str[1:].astype(int) - 1
        part = part.drop_duplicates(subset=[bin_col]).sort_values('local_index')
        return part

    family_offsets = {}
    family_name_to_global = {}
    global_index_to_name = {}
    total_real_slots = 0
    rows = []

    for family, (name_col, bin_col) in FAMILY_META.items():
        entries = extract_family_entries(suzuki_dict_df, family, name_col, bin_col)
        family_offsets[family] = total_real_slots
        family_name_to_global[family] = {}
        for _, row in entries.iterrows():
            global_index = total_real_slots + int(row['local_index'])
            name = str(row[name_col]).strip()
            family_name_to_global[family][name] = global_index
            global_index_to_name[global_index] = name
        rows.append({
            'family': family,
            'num_labels': len(entries),
            'global_start': total_real_slots,
            'global_end': total_real_slots + len(entries) - 1,
        })
        total_real_slots += len(entries)

    reserved_unknown_index = total_real_slots
    null_condition_index = {family: total_real_slots + 1 + i for i, family in enumerate(FAMILY_META)}
    class_num = total_real_slots + 1 + len(FAMILY_META)

    summary_df = pd.DataFrame(rows)
    print('family summary:')
    display(summary_df)
    print('total_real_slots      =', total_real_slots)
    print('reserved_unknown_index =', reserved_unknown_index)
    print('null_condition_index   =', null_condition_index)
    print('class_num              =', class_num)
    '''),
    md('''
    ## 3. 解析反应 SMILES

    训练脚本最终把三种分子作为三个独立输入：

    - `Reactant1`
    - `Reactant2`
    - `Product`

    这正对应 `train.py` 里的：

    ```python
    smiles_col=['Reactant1', 'Reactant2', 'Product']
    ```

    因此我们要先把一条 `reaction_smiles` 拆成这三列，然后做基础标准化。

    ### 源码对应

    - `source_repos/reaction-gcnn/train.py`
    - `source_repos/reaction-gcnn/dataset/suzuki_data_frame_parser.py`
    '''),
    code('''
    # ====== Step 3: 解析 reaction_smiles 并标准化 ======
    def canonicalize_smiles(smiles):
        if pd.isna(smiles) or not str(smiles).strip():
            return None
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            raise ValueError(f'无法解析 SMILES: {smiles}')
        return Chem.MolToSmiles(mol, canonical=True)

    def split_reaction_smiles(reaction_smiles):
        reactants, product = reaction_smiles.split('>>')
        reactant_parts = [part for part in reactants.split('.') if part]
        if len(reactant_parts) == 1:
            reactant_parts.append(None)
        if len(reactant_parts) != 2:
            raise ValueError(f'教学版当前假设每条反应有两个底物，实际得到: {reactant_parts}')
        return {
            'Reactant1': canonicalize_smiles(reactant_parts[0]),
            'Reactant2': canonicalize_smiles(reactant_parts[1]) if reactant_parts[1] else None,
            'Product': canonicalize_smiles(product),
        }

    split_df = demo_df['reaction_smiles'].apply(split_reaction_smiles).apply(pd.Series)
    parsed_df = pd.concat([demo_df, split_df], axis=1)
    parsed_df[['reaction_id', 'Reactant1', 'Reactant2', 'Product', 'yield']]
    '''),
    md('''
    ## 4. 条件名称编码为仓库风格的标签字段

    原 parser 期待的标签列不是名称，而是类似：

    - `M = "[0]"`
    - `L = "[28]"`
    - `S = "[86, 94]"`

    也就是说：

    1. 名称先根据字典转成全局索引
    2. 每个条件家族允许多个索引
    3. 最终写成字符串形式的 Python list

    下面这一步会同时导出一个教学版的 `demo_repo_style.csv`，方便你对照原仓库输入格式。
    '''),
    code('''
    # ====== Step 4: 条件名称 -> 全局索引 -> 仓库风格字段 ======
    def parse_condition_names(raw_value):
        if pd.isna(raw_value) or not str(raw_value).strip():
            return []
        return [item.strip() for item in str(raw_value).split('|') if item.strip()]

    def encode_condition_names(row):
        encoded = {}
        for raw_col, family in RAW_COL_TO_FAMILY.items():
            names = parse_condition_names(row[raw_col])
            global_indices = []
            for name in names:
                if name not in family_name_to_global[family]:
                    raise KeyError(f'{family} 家族中找不到条件名称: {name}')
                global_indices.append(family_name_to_global[family][name])
            encoded[family] = global_indices
        return encoded

    def row_to_repo_style(row):
        encoded = encode_condition_names(row)
        result = {
            'id': row['reaction_id'],
            'Yield': row['yield'],
            'Reactant1': row['Reactant1'],
            'Reactant2': row['Reactant2'],
            'Product': row['Product'],
        }
        for family in FAMILY_META:
            result[family] = str(encoded[family])
        return result

    repo_style_df = parsed_df.apply(row_to_repo_style, axis=1, result_type='expand')
    repo_style_df.to_csv(REPO_STYLE_CSV, sep=';', index=False)

    print('导出的仓库风格文件:', REPO_STYLE_CSV.relative_to(PROJECT_ROOT))
    repo_style_df
    '''),
    md('''
    ## 5. 把分子转换为 RelGCN 需要的图结构

    `models/relgcn.py` 的输入核心是两部分：

    - `atom_ids`: 每个原子的原子序数
    - `adj`: 形状为 `(num_edge_type, num_nodes, num_nodes)` 的关系邻接张量

    对于这个仓库，`num_edge_type = 4`，分别表示：

    1. 单键
    2. 双键
    3. 三键
    4. 芳香键

    ### 源码对应

    - `source_repos/reaction-gcnn/models/relgcn.py`
    - `source_repos/reaction-gcnn/train_attention_model.py` 中 `num_edge_type = 4`
    '''),
    code('''
    # ====== Step 5: SMILES -> RelGCN 图张量 ======
    BOND_TYPE_TO_CHANNEL = {
        Chem.rdchem.BondType.SINGLE: 0,
        Chem.rdchem.BondType.DOUBLE: 1,
        Chem.rdchem.BondType.TRIPLE: 2,
        Chem.rdchem.BondType.AROMATIC: 3,
    }

    def smiles_to_relgcn_graph(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f'无法解析分子: {smiles}')

        atom_ids = np.array([atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype=np.int64)
        num_atoms = len(atom_ids)
        adj = np.zeros((4, num_atoms, num_atoms), dtype=np.float32)

        for bond in mol.GetBonds():
            begin = bond.GetBeginAtomIdx()
            end = bond.GetEndAtomIdx()
            channel = BOND_TYPE_TO_CHANNEL.get(bond.GetBondType())
            if channel is None:
                continue
            adj[channel, begin, end] = 1.0
            adj[channel, end, begin] = 1.0

        mask = np.ones(num_atoms, dtype=np.float32)
        return {
            'atom_ids': atom_ids,
            'adj': adj,
            'mask': mask,
            'num_atoms': num_atoms,
        }

    graph_example = smiles_to_relgcn_graph(repo_style_df.loc[0, 'Reactant1'])
    print('example smiles:', repo_style_df.loc[0, 'Reactant1'])
    print('atom_ids shape:', graph_example['atom_ids'].shape)
    print('adj shape     :', graph_example['adj'].shape)
    print('num_atoms     :', graph_example['num_atoms'])
    print('atom_ids      :', graph_example['atom_ids'])
    '''),
    md('''
    ## 6. 组装成 ARGCN 教学版样本与 batch

    到这里，我们已经有了：

    - 三个分子的图结构
    - 一个样本的多热标签向量

    剩下的事情就是把多个样本 pad 成 batch。这一步相当于教学版对原仓库 `concat_mols` 的手工复现。

    ### 输出张量说明

    | 键 | 含义 |
    |----|------|
    | `atom_ids_1 / atom_ids_2 / atom_ids_3` | 三个分子的原子编号张量 |
    | `adjs_1 / adjs_2 / adjs_3` | 三个分子的关系邻接张量 |
    | `mask_1 / mask_2 / mask_3` | pad 位置掩码 |
    | `labels` | 长度为 119 的多热向量 |
    | `yields` | 反应收率 |
    '''),
    code('''
    # ====== Step 6: 构造样本对象并拼 batch ======
    def encode_multihot_target(encoded_conditions):
        target = np.zeros(class_num, dtype=np.float32)
        for family in FAMILY_META:
            indices = encoded_conditions[family]
            if indices:
                target[indices] = 1.0
            else:
                target[null_condition_index[family]] = 1.0
        return target

    def build_argcn_item(row):
        encoded = encode_condition_names(row)
        return {
            'reaction_id': row['reaction_id'],
            'yield_value': float(row['yield']),
            'graphs': [
                smiles_to_relgcn_graph(row['Reactant1']),
                smiles_to_relgcn_graph(row['Reactant2']),
                smiles_to_relgcn_graph(row['Product']),
            ],
            'encoded_conditions': encoded,
            'label_vector': encode_multihot_target(encoded),
        }

    def pad_atom_ids(atom_ids, max_atoms):
        padded = np.zeros(max_atoms, dtype=np.int64)
        padded[:len(atom_ids)] = atom_ids
        return padded

    def pad_mask(mask, max_atoms):
        padded = np.zeros(max_atoms, dtype=np.float32)
        padded[:len(mask)] = mask
        return padded

    def pad_adj(adj, max_atoms):
        padded = np.zeros((adj.shape[0], max_atoms, max_atoms), dtype=np.float32)
        n = adj.shape[1]
        padded[:, :n, :n] = adj
        return padded

    def collate_argcn_batch(items):
        batch = {'reaction_ids': [item['reaction_id'] for item in items]}
        for mol_idx in range(3):
            max_atoms = max(item['graphs'][mol_idx]['num_atoms'] for item in items)
            batch[f'atom_ids_{mol_idx+1}'] = torch.tensor(
                np.stack([pad_atom_ids(item['graphs'][mol_idx]['atom_ids'], max_atoms) for item in items]),
                dtype=torch.long,
            )
            batch[f'adjs_{mol_idx+1}'] = torch.tensor(
                np.stack([pad_adj(item['graphs'][mol_idx]['adj'], max_atoms) for item in items]),
                dtype=torch.float32,
            )
            batch[f'mask_{mol_idx+1}'] = torch.tensor(
                np.stack([pad_mask(item['graphs'][mol_idx]['mask'], max_atoms) for item in items]),
                dtype=torch.float32,
            )
        batch['labels'] = torch.tensor(np.stack([item['label_vector'] for item in items]), dtype=torch.float32)
        batch['yields'] = torch.tensor([item['yield_value'] for item in items], dtype=torch.float32)
        return batch

    items = [build_argcn_item(row) for _, row in parsed_df.iterrows()]
    batch = collate_argcn_batch(items)

    print('reaction_ids:', batch['reaction_ids'])
    for key, value in batch.items():
        if isinstance(value, torch.Tensor):
            print(f'{key:12s} -> shape={tuple(value.shape)}, dtype={value.dtype}')
    '''),
    md('''
    ## 完整流程总结

    ### 数据处理管线回顾

    ```text
    demo_data.csv
        ↓
    解析 reaction_smiles
        ↓
    Reactant1 / Reactant2 / Product
        ↓
    条件名称映射到全局索引
        ↓
    构造 119 维多热标签
        ↓
    每个分子转成 (atom_ids, relation_adjs, mask)
        ↓
    pad 成 batch
    ```

    ### 教学版与原仓库的差异

    - 原仓库通过 `chainer-chemistry` 的 preprocessor 自动完成部分图特征提取。
    - 教学版显式地把每一步写出来，便于观察张量结构。
    - 原仓库里还包含缓存、更多数据集分支、训练/验证拆分等工程细节，这里暂时省略。

    下一步建议打开 `3_模型展示.ipynb`，把这些张量如何进入 Torch 版 ARGCN 模型看清楚。
    '''),
]

model_cells = [
    md('''
    # ARGCN 模型展示教程

    ## 概述

    本 notebook 用 **PyTorch** 复现 `reaction-gcnn` 中 attention 版本的核心推理路径。

    这里说的 **ARGCN**，可以理解为：

    - 共享的 **RelGCN** 分子编码器
    - 面向三分子集合的 **reaction-level attention readout**
    - 面向条件空间的 **多标签分类头**

    它对应原仓库这三部分：

    - `source_repos/reaction-gcnn/models/relgcn.py`
    - `source_repos/reaction-gcnn/models/attention_readout.py`
    - `source_repos/reaction-gcnn/train_attention_model.py`

    ### 本 notebook 内容

    | 章节 | 内容 |
    |------|------|
    | 1 | 准备与数据 batch |
    | 2 | Torch 版 RelGCN 编码器 |
    | 3 | Torch 版 reaction attention readout |
    | 4 | 组装 ARGCN 顶层模型 |
    | 5 | 跑一次前向推理 |
    | 6 | 用极小样本做 overfit 演示并解码预测 |
    '''),
    code('''
    # ====== Step 0: 导入与路径 ======
    import os
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from rdkit import Chem

    def find_project_root(start=None):
        here = Path(start or os.getcwd()).resolve()
        for path in [here, *here.parents]:
            if (path / 'teaching_demos').exists() and (path / 'source_repos').exists():
                return path
        raise RuntimeError('未找到项目根目录')

    PROJECT_ROOT = find_project_root()
    TUTORIAL_DIR = PROJECT_ROOT / 'teaching_demos' / '4.reaction_condition_tutorial' / 'ARGCN'
    REPO_DIR = PROJECT_ROOT / 'source_repos' / 'reaction-gcnn'
    DEMO_DATA = TUTORIAL_DIR / 'data' / 'demo_data.csv'
    DICT_PATH = REPO_DIR / 'data' / 'all_dictionaries' / 'suzuki_dict.csv'

    print('PROJECT_ROOT =', PROJECT_ROOT)
    print('DEMO_DATA    =', DEMO_DATA)
    print('DICT_PATH    =', DICT_PATH)
    '''),
    md('''
    ## 1. 先准备和数据 notebook 一致的 batch

    为了让本 notebook 自包含，下面会用与 `2_数据处理.ipynb` 相同的逻辑，把 demo 数据直接变成一个小 batch。

    ### 这一步的目标

    后面模型只关心这些张量：

    - `atom_ids_1 / adjs_1 / mask_1`
    - `atom_ids_2 / adjs_2 / mask_2`
    - `atom_ids_3 / adjs_3 / mask_3`
    - `labels`
    '''),
    code('''
    # ====== Step 1: 构造教学 batch ======
    demo_df = pd.read_csv(DEMO_DATA)
    suzuki_dict_df = pd.read_csv(DICT_PATH)

    FAMILY_META = {
        'M': ('metal_name', 'metal_bin'),
        'L': ('ligand_name', 'ligand_bin'),
        'B': ('base_name', 'base_bin'),
        'S': ('solvent_name', 'solvent_bin'),
        'A': ('additive_name', 'additive_bin'),
    }
    RAW_COL_TO_FAMILY = {
        'metal': 'M',
        'ligand': 'L',
        'base': 'B',
        'solvent': 'S',
        'additive': 'A',
    }
    BOND_TYPE_TO_CHANNEL = {
        Chem.rdchem.BondType.SINGLE: 0,
        Chem.rdchem.BondType.DOUBLE: 1,
        Chem.rdchem.BondType.TRIPLE: 2,
        Chem.rdchem.BondType.AROMATIC: 3,
    }

    def extract_family_entries(df, family, name_col, bin_col):
        part = df[[name_col, bin_col]].dropna().copy()
        part[bin_col] = part[bin_col].astype(str)
        part = part[part[bin_col].str.startswith(family)]
        part['local_index'] = part[bin_col].str[1:].astype(int) - 1
        part = part.drop_duplicates(subset=[bin_col]).sort_values('local_index')
        return part

    family_name_to_global = {}
    global_index_to_name = {}
    total_real_slots = 0
    for family, (name_col, bin_col) in FAMILY_META.items():
        entries = extract_family_entries(suzuki_dict_df, family, name_col, bin_col)
        family_name_to_global[family] = {}
        for _, row in entries.iterrows():
            global_index = total_real_slots + int(row['local_index'])
            name = str(row[name_col]).strip()
            family_name_to_global[family][name] = global_index
            global_index_to_name[global_index] = name
        total_real_slots += len(entries)

    reserved_unknown_index = total_real_slots
    null_condition_index = {family: total_real_slots + 1 + i for i, family in enumerate(FAMILY_META)}
    class_num = total_real_slots + 1 + len(FAMILY_META)

    family_slices = {}
    start = 0
    for family, (name_col, bin_col) in FAMILY_META.items():
        size = len(extract_family_entries(suzuki_dict_df, family, name_col, bin_col))
        family_slices[family] = slice(start, start + size)
        start += size

    def canonicalize_smiles(smiles):
        if pd.isna(smiles) or not str(smiles).strip():
            return None
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            raise ValueError(f'无法解析 SMILES: {smiles}')
        return Chem.MolToSmiles(mol, canonical=True)

    def split_reaction_smiles(reaction_smiles):
        reactants, product = reaction_smiles.split('>>')
        reactant_parts = [part for part in reactants.split('.') if part]
        if len(reactant_parts) == 1:
            reactant_parts.append(None)
        if len(reactant_parts) != 2:
            raise ValueError(f'教学版假设两个底物，得到: {reactant_parts}')
        return {
            'Reactant1': canonicalize_smiles(reactant_parts[0]),
            'Reactant2': canonicalize_smiles(reactant_parts[1]) if reactant_parts[1] else None,
            'Product': canonicalize_smiles(product),
        }

    def parse_condition_names(raw_value):
        if pd.isna(raw_value) or not str(raw_value).strip():
            return []
        return [item.strip() for item in str(raw_value).split('|') if item.strip()]

    def encode_condition_names(row):
        encoded = {}
        for raw_col, family in RAW_COL_TO_FAMILY.items():
            names = parse_condition_names(row[raw_col])
            indices = [family_name_to_global[family][name] for name in names]
            encoded[family] = indices
        return encoded

    def encode_multihot_target(encoded_conditions):
        target = np.zeros(class_num, dtype=np.float32)
        for family in FAMILY_META:
            if encoded_conditions[family]:
                target[encoded_conditions[family]] = 1.0
            else:
                target[null_condition_index[family]] = 1.0
        return target

    def smiles_to_relgcn_graph(smiles):
        mol = Chem.MolFromSmiles(smiles)
        atom_ids = np.array([atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype=np.int64)
        n = len(atom_ids)
        adj = np.zeros((4, n, n), dtype=np.float32)
        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()
            channel = BOND_TYPE_TO_CHANNEL[bond.GetBondType()]
            adj[channel, i, j] = 1.0
            adj[channel, j, i] = 1.0
        mask = np.ones(n, dtype=np.float32)
        return {'atom_ids': atom_ids, 'adj': adj, 'mask': mask, 'num_atoms': n}

    parsed_df = pd.concat([demo_df, demo_df['reaction_smiles'].apply(split_reaction_smiles).apply(pd.Series)], axis=1)

    def build_item(row):
        encoded = encode_condition_names(row)
        return {
            'reaction_id': row['reaction_id'],
            'graphs': [
                smiles_to_relgcn_graph(row['Reactant1']),
                smiles_to_relgcn_graph(row['Reactant2']),
                smiles_to_relgcn_graph(row['Product']),
            ],
            'labels': encode_multihot_target(encoded),
            'encoded_conditions': encoded,
        }

    items = [build_item(row) for _, row in parsed_df.iterrows()]

    def pad_atom_ids(atom_ids, max_atoms):
        arr = np.zeros(max_atoms, dtype=np.int64)
        arr[:len(atom_ids)] = atom_ids
        return arr

    def pad_mask(mask, max_atoms):
        arr = np.zeros(max_atoms, dtype=np.float32)
        arr[:len(mask)] = mask
        return arr

    def pad_adj(adj, max_atoms):
        arr = np.zeros((adj.shape[0], max_atoms, max_atoms), dtype=np.float32)
        n = adj.shape[1]
        arr[:, :n, :n] = adj
        return arr

    def collate(items):
        batch = {'reaction_ids': [item['reaction_id'] for item in items]}
        for mol_idx in range(3):
            max_atoms = max(item['graphs'][mol_idx]['num_atoms'] for item in items)
            batch[f'atom_ids_{mol_idx+1}'] = torch.tensor(
                np.stack([pad_atom_ids(item['graphs'][mol_idx]['atom_ids'], max_atoms) for item in items]),
                dtype=torch.long,
            )
            batch[f'adjs_{mol_idx+1}'] = torch.tensor(
                np.stack([pad_adj(item['graphs'][mol_idx]['adj'], max_atoms) for item in items]),
                dtype=torch.float32,
            )
            batch[f'mask_{mol_idx+1}'] = torch.tensor(
                np.stack([pad_mask(item['graphs'][mol_idx]['mask'], max_atoms) for item in items]),
                dtype=torch.float32,
            )
        batch['labels'] = torch.tensor(np.stack([item['labels'] for item in items]), dtype=torch.float32)
        return batch

    batch = collate(items)
    for key, value in batch.items():
        if isinstance(value, torch.Tensor):
            print(f'{key:12s} -> shape={tuple(value.shape)}')
    '''),
    md('''
    ## 2. Torch 版 RelGCN 编码器

    原仓库的 `models/relgcn.py` 做了三件事：

    1. 把 `atom_ids` 嵌入成连续向量。
    2. 对每一种边类型分别做消息传递。
    3. 在每一层后接 `tanh`。

    下面的教学版 `RelGCNLayer` 不依赖 `chainer-chemistry`，但保留同样的核心公式：

    $$
    H^{(l+1)} = \tanh\left(W_0 H^{(l)} + \sum_r W_r (\hat{A}_r H^{(l)})\right)
    $$

    其中 $r$ 表示不同类型的化学键关系。
    '''),
    code('''
    # ====== Step 2: Torch 版 RelGCN ======
    def rescale_relation_adjacency(adj, mask=None):
        degree = adj.sum(dim=1).sum(dim=-1)  # [B, N]
        degree = torch.where(degree > 0, degree, torch.ones_like(degree))
        adj = adj / degree[:, None, :, None]
        if mask is not None:
            edge_mask = mask[:, None, :, None] * mask[:, None, None, :]
            adj = adj * edge_mask
        return adj

    class RelGCNLayer(nn.Module):
        def __init__(self, hidden_dim, num_relations=4):
            super().__init__()
            self.self_linear = nn.Linear(hidden_dim, hidden_dim)
            self.rel_linears = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(num_relations)])

        def forward(self, h, adj, mask=None):
            adj = rescale_relation_adjacency(adj, mask)
            messages = self.self_linear(h)
            for rel_id, linear in enumerate(self.rel_linears):
                neigh = torch.matmul(adj[:, rel_id], h)
                messages = messages + linear(neigh)
            out = torch.tanh(messages)
            if mask is not None:
                out = out * mask.unsqueeze(-1)
            return out

    class RelGCNEncoder(nn.Module):
        def __init__(self, max_atomic_num=100, hidden_dim=64, num_layers=3, num_relations=4):
            super().__init__()
            self.embedding = nn.Embedding(max_atomic_num + 1, hidden_dim, padding_idx=0)
            self.layers = nn.ModuleList([RelGCNLayer(hidden_dim, num_relations) for _ in range(num_layers)])

        def forward(self, atom_ids, adjs, mask):
            h = self.embedding(atom_ids)
            for layer in self.layers:
                h = layer(h, adjs, mask)
            return h
    '''),
    md('''
    ## 3. reaction-level attention readout

    这是 `train.py` 和 `train_attention_model.py` 的关键差异。

    - 基础版 `train.py`: 每个分子先做图级读出，然后把 3 个分子向量直接拼接。
    - attention 版 `train_attention_model.py`: 先保留每个节点嵌入，再做**跨三分子的节点级注意力聚合**。

    原仓库 `models/attention_readout.py` 的逻辑可以拆成两级门控：

    1. 对每个分子内节点做门控读出。
    2. 把三个分子的节点拼在一起，再做 reaction-level attention。

    下面的教学实现保留了这个思路，并额外输出每个节点的 attention 强度，方便观察模型关注点。
    '''),
    code('''
    # ====== Step 3: Torch 版 reaction attention readout ======
    class ReactionAttentionReadout(nn.Module):
        def __init__(self, node_dim=64, readout_dim=128):
            super().__init__()
            self.i_proj = nn.Linear(node_dim, readout_dim)
            self.j_proj = nn.Linear(node_dim, readout_dim)
            self.att_proj = nn.Linear(readout_dim, readout_dim * 3)
            self.out_proj = nn.Linear(readout_dim, readout_dim * 3)

        def molecule_gate(self, nodes, mask):
            gated = torch.sigmoid(self.i_proj(nodes)) * torch.tanh(self.j_proj(nodes))
            gated = gated * mask.unsqueeze(-1)
            return gated

        def forward(self, node_sets, mask_sets):
            gated_sets = [self.molecule_gate(nodes, mask) for nodes, mask in zip(node_sets, mask_sets)]
            concat_nodes = torch.cat(gated_sets, dim=1)
            concat_mask = torch.cat(mask_sets, dim=1)

            att = torch.sigmoid(self.att_proj(concat_nodes))
            values = torch.tanh(self.out_proj(concat_nodes))
            weighted = att * values * concat_mask.unsqueeze(-1)
            reaction_embedding = weighted.sum(dim=1)

            node_attention = att.mean(dim=-1) * concat_mask
            return reaction_embedding, node_attention
    '''),
    md('''
    ## 4. 组装顶层 ARGCN 模型

    现在把部件按 `train_attention_model.py` 的顺序拼起来：

    ```text
    Reactant1 graph  ┐
    Reactant2 graph  ├─ shared RelGCN encoder ─┐
    Product graph    ┘                         ├─ reaction attention ─ MLP ─ logits
                                               ┘
    ```

    注意这里的“共享”非常重要：三个分子共用同一个分子编码器参数。
    '''),
    code('''
    # ====== Step 4: ARGCN 顶层模型 ======
    class ARGCNPredictor(nn.Module):
        def __init__(self, max_atomic_num=100, hidden_dim=64, num_layers=3, num_relations=4, class_num=119):
            super().__init__()
            self.encoder = RelGCNEncoder(
                max_atomic_num=max_atomic_num,
                hidden_dim=hidden_dim,
                num_layers=num_layers,
                num_relations=num_relations,
            )
            self.readout = ReactionAttentionReadout(node_dim=hidden_dim, readout_dim=128)
            self.classifier = nn.Sequential(
                nn.Linear(128 * 3, 128),
                nn.ReLU(),
                nn.Linear(128, class_num),
            )

        def forward(self, batch):
            node_sets = []
            mask_sets = []
            for idx in (1, 2, 3):
                atom_ids = batch[f'atom_ids_{idx}']
                adjs = batch[f'adjs_{idx}']
                mask = batch[f'mask_{idx}']
                node_repr = self.encoder(atom_ids, adjs, mask)
                node_sets.append(node_repr)
                mask_sets.append(mask)
            reaction_embedding, node_attention = self.readout(node_sets, mask_sets)
            logits = self.classifier(reaction_embedding)
            return logits, node_attention
    '''),
    md('''
    ## 5. 跑一次前向推理，并观察张量形状

    这一步的目标很简单：

    - 确认模型能吃进 `2_数据处理.ipynb` 里构造的 batch
    - 看清每个张量的维度变化
    - 为后面的 tiny overfit 做准备
    '''),
    code('''
    # ====== Step 5: 前向推理演示 ======
    torch.manual_seed(7)

    model = ARGCNPredictor(class_num=class_num)
    logits, node_attention = model(batch)

    print('logits shape        :', tuple(logits.shape))
    print('node_attention shape:', tuple(node_attention.shape))
    print('batch labels shape  :', tuple(batch['labels'].shape))
    print('first sample logits (first 12 dims):')
    print(logits[0, :12].detach())
    '''),
    md('''
    ## 6. 用极小样本做一个 overfit 演示

    为什么这里要加一个小训练循环？

    因为随机初始化下的 logits 没什么解释价值。用 3 条样本做几十到一百多步的小规模拟合，可以让你看到：

    1. 多标签 BCE 损失如何工作
    2. 模型能否记住当前这 3 条反应
    3. 每个条件家族的 top 预测如何解码回具体名称

    这不是正式训练，只是帮助理解推理结果。
    '''),
    code('''
    # ====== Step 6: tiny overfit + 条件解码 ======
    torch.manual_seed(7)
    model = ARGCNPredictor(class_num=class_num)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()

    for step in range(120):
        optimizer.zero_grad()
        logits, node_attention = model(batch)
        loss = loss_fn(logits, batch['labels'])
        loss.backward()
        optimizer.step()
        if step % 20 == 0 or step == 119:
            print(f'step={step:03d}  loss={loss.item():.4f}')

    def decode_predictions(prob_vector, topk=2):
        decoded = {}
        for family, family_slice in family_slices.items():
            family_probs = prob_vector[family_slice]
            top_indices = torch.topk(family_probs, k=min(topk, len(family_probs))).indices.tolist()
            decoded[family] = [
                {
                    'global_index': family_slice.start + idx,
                    'name': global_index_to_name[family_slice.start + idx],
                    'prob': float(family_probs[idx]),
                }
                for idx in top_indices
            ]
            decoded[f'{family}_null_prob'] = float(prob_vector[null_condition_index[family]])
        return decoded

    with torch.no_grad():
        logits, node_attention = model(batch)
        probs = torch.sigmoid(logits)

    for sample_idx, reaction_id in enumerate(batch['reaction_ids']):
        print('\n' + '=' * 80)
        print('reaction_id:', reaction_id)
        decoded = decode_predictions(probs[sample_idx], topk=2)
        for family in FAMILY_META:
            print(f'[{family}] top predictions:')
            for item in decoded[family]:
                print('  ', item)
            print(f'  null prob: {decoded[f"{family}_null_prob"]:.4f}')

        mask_lengths = [int(batch[f'mask_{idx}'][sample_idx].sum().item()) for idx in (1, 2, 3)]
        ranges = []
        start = 0
        for length in mask_lengths:
            ranges.append((start, start + length))
            start += length
        mol_names = ['Reactant1', 'Reactant2', 'Product']
        total_attention = node_attention[sample_idx].sum().item() + 1e-8
        for mol_name, (s, e) in zip(mol_names, ranges):
            score = float(node_attention[sample_idx, s:e].sum().item() / total_attention)
            print(f'  attention share - {mol_name}: {score:.3f}')
    '''),
    md('''
    ## 7. 总结

    ### 本 notebook 完成了什么

    - 用 **Torch** 复现了 `reaction-gcnn` attention 版本的主干结构。
    - 解释了为什么这个模型是“共享分子编码器 + 三分子级注意力聚合 + 多标签分类头”。
    - 用一个极小样本 overfit 演示，把前向推理、损失函数和结果解码串了起来。

    ### 教学版与原仓库的差异

    - 原仓库依赖 `chainer-chemistry` 的现成组件；教学版全部显式写成 Torch 模块。
    - 原仓库还有更多训练、缓存、评估脚本；这里仅保留最核心的编码与推理路径。
    - 教学版额外返回了 attention 强度，便于解释模型关注点。

    到这里，你已经可以把 `reaction-gcnn` 的 attention 版本当成一个可读的 Torch 教程来理解，而不是只能黑盒调用旧代码。
    '''),
]

notebooks = {
    '1_环境配置.ipynb': notebook(env_cells),
    '2_数据处理.ipynb': notebook(data_cells),
    '3_模型展示.ipynb': notebook(model_cells),
}

for filename, payload in notebooks.items():
    path = TUTORIAL_DIR / filename
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=1))
    print('wrote', path)

print('done')

wrote /home/xiaoruiwang/data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/4.reaction_condition_tutorial/ARGCN/1_环境配置.ipynb
wrote /home/xiaoruiwang/data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/4.reaction_condition_tutorial/ARGCN/2_数据处理.ipynb
wrote /home/xiaoruiwang/data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/4.reaction_condition_tutorial/ARGCN/3_模型展示.ipynb
done


<>:1016: SyntaxWarning: invalid escape sequence '\l'
<>:1016: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_1187471/2441829019.py:1016: SyntaxWarning: invalid escape sequence '\l'
  H^{(l+1)} = \tanh\left(W_0 H^{(l)} + \sum_r W_r (\hat{A}_r H^{(l)})\right)
